Pontificia Universidad Católica de Chile <br>
Departamento de Ciencia de la Computación <br>

<br>

<center>
    <h2> T1: Parte 1 </h2>
    <h1> Procesamiento de Datos Masivos (IIC2440)</h1>
 
</center>




---

## Introducción

El presente notebook implementa un pipeline ETL reproducible orientado al diseño y construcción de un data warehouse sobre una colección masiva de artículos de noticias chilenas. El objetivo principal es transformar datos crudos en una estructura analítica eficiente, siguiendo principios de modelamiento dimensional y buenas prácticas de ingeniería de datos. Se adopta un esquema estrella (star schema), compuesto por una tabla de hechos (fact_news) y múltiples tablas de dimensiones (dim_date, dim_source, dim_region), permitiendo facilitar consultas analíticas y asegurar consistencia en los datos.

El pipeline ETL se estructura en distintas etapas: extracción de los datos desde un archivo CSV, limpieza y normalización de los registros, construcción de dimensiones, asignación de llaves foráneas y generación de la tabla de hechos final. Adicionalmente, se implementa un esquema de almacenamiento particionado por año y mes, siguiendo prácticas comunes en arquitecturas tipo Data Lake.

![Pipeline ETL](pipeline.jpg)


## Librerías

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import pyarrow
import re
import os

## Carga base datos

In [5]:
noticias_df = pd.read_csv("noticias_chile_2023_2025_unique_article_id.csv")
# Tamaño inicial
print(f"El tamaño del dataset es de {noticias_df.shape[0]} registros y {noticias_df.shape[1]} variables.")

El tamaño del dataset es de 1467310 registros y 6 variables.


In [6]:
noticias_df.head()

,article_id,title,body,publish_date,source,country
0,a84966b0b8f51e7c989c0f85f41a2ca7a94243a14492ab...,Cierran Zona Zero Por Vender Alcohol Sin Tener...,"Delegado presidencial Daniel Quinteros dijo: ""...",2023-04-24,diariolongino.cl,Chile
1,928d2ad41f272019aa9d9016d15c194626ca27f2283372...,Página 7 | El Mercurio de Valparaíso,HOMENAJE. Y el comando de aviación más antiguo...,2023-09-17,mercuriovalpo.cl,Chile
2,c7c2ddf848419b1c0421d5383d925330d2b4ee37c1e9ca...,"Aparece el PCB de un RTX 5090, confirmaría 32G...",La PCB de la GPU GeForce RTX 5090 de NVIDIA ac...,2024-12-25,madboxpc.com,Chile
3,3e231efebb445624ce66cac109b4c5d9be5468ebc1ae9a...,La IA No Llegó Para Quedarse Con Tu Trabajo | ...,Los avances tecnológicos han cambiado a la hum...,2023-09-14,diariolongino.cl,Chile
4,a4fd0786a6e0f11339f47f8d0c8d4085fdf9b0493fdb63...,Trabajador fallece por inhalación de monóxido ...,Un trabajador falleció la tarde de este lunes ...,2025-12-09,miradiols.cl,Chile


In [7]:
noticias_df.isnull().sum()

article_id       0
title           58
body             0
publish_date     0
source           0
country          0
dtype: int64

## Limpieza y normalización

In [2]:
def limpiar_dataset(df):
    """
    Limpieza de datos nulos y cambio
    de fechas a formato estándar
    """
    # Cambiar data nula (no eliminar por recuento final)
    df['title'] = df['title'].fillna("sin titulo")
    df['body'] = df['body'].fillna("sin cuerpo")
    df['source'] = df['source'].fillna("fuente desconocida")
    
    # Formato fechas
    df['publish_date'] = pd.to_datetime(df['publish_date'], errors='coerce')
    df_limpio = df.dropna(subset=['publish_date'])
    
    return df_limpio


def normalizar_texto(text):
    """
    Normaliza una str: 
    minúsculas, quita acentos y caracteres especiales.
    (insensible a mayúsculas y acentos).
    """
    text = str(text) if text is not None else "" # errores numéricos
    s= text.lower()
    n= unicodedata.normalize('NFKD', s)  
    res = ''.join([c for c in n if not unicodedata.combining(c)])  
    
    return res

## Contrucción de dimensiones y Fact table

In [3]:
def build_date_dim(df):
    """
    Crea la dimensión de tiempo a partir de las fechas del dataset
    """
    # rango max de fechas del df 
    min_date = df['publish_date'].min()
    max_date = df['publish_date'].max()
    
    date_range = pd.date_range(start=min_date, end=max_date, freq='D')
    dim_date = pd.DataFrame({'full_date': date_range})
    
    # adición de columnas a la dimensión
    dim_date['date_id'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int) # LLAVE SIN DUPLICADOS 
    dim_date['year'] = dim_date['full_date'].dt.year
    #dim_date['month'] = dim_date['full_date'].dt.month
    dim_date['month'] = dim_date['full_date'].dt.month.map('{:02d}'.format) # formato de almacenamiento particionado (2 -> 02)

    dim_date['day'] = dim_date['full_date'].dt.day
    dim_date['day_of_week'] = dim_date['full_date'].dt.day_name() 
    
    return dim_date

In [4]:
def build_source_dim(df):
    """
    Crea la dimensión de fuentes de noticias a partir de los datos limpios
    """
    fuentes_unicas = df['source'].unique()
    dim_source = pd.DataFrame({'source_name': fuentes_unicas})
    dim_source['source_id'] = range(1, len(dim_source) + 1) # sk unica sin repetir

    return dim_source

In [ ]:
def dicc_regiones():
    """
    Función que define la dim_regiones
    con cada región asociada a un id (retorna dim date).
    Además, se define el diccionario
    de alias para cada una para posterior
    uso en el procesamiento del pipeline (retorna dicc de alias)
    """

    regiones = [
        "Arica y Parinacota",
        "Tarapaca",
        "Antofagasta",
        "Atacama",
        "Coquimbo",
        "Valparaiso",
        "Metropolitana",
        "Ohiggins",
        "Maule",
        "Nuble",
        "Biobio",
        "La Araucania",
        "Los Rios",
        "Los Lagos",
        "Aysen",
        "Magallanes",
        "Desconocida"
    ]

    # construir dimensión región
    dim_region = pd.DataFrame({
        "region_key": range(1, len(regiones) + 1),
        "region_name": regiones
    })

    # Alias 
    region_aliases = {
        1: ["arica", "parinacota"],
        2: ["tarapaca", "iquique", "alto hospicio"],
        3: ["antofagasta", "calama", "tocopilla"],
        4: ["atacama", "copiapo", "vallenar"],
        5: ["coquimbo", "la serena", "ovalle"],
        6: ["valparaiso", "vina del mar", "quilpue", "san antonio"],
        7: ["metropolitana", "santiago", "rm", "region metropolitana", "capital"],
        8: ["ohiggins", "rancagua", "san fernando"],
        9: ["maule", "talca", "curico", "linares"],
        10: ["nuble", "chillan"],
        11: ["biobio", "concepcion", "talcahuano", "los angeles"],
        12: ["araucania", "temuco"],
        13: ["los rios", "valdivia"],
        14: ["los lagos", "puerto montt", "osorno", "castro"],
        15: ["aysen", "coihaique", "coyhaique"],
        16: ["magallanes", "punta arenas"],
        17: []  # desconocida
    }

    return dim_region, region_aliases
    

def construir_patrones(region_aliases):
    """Compila las regex una sola vez para todo el proceso"""
    patrones = {}
    for key, aliases in region_aliases.items():
        if key == 17: continue 
        pattern_str = r'\b(?:' + '|'.join(map(re.escape, aliases)) + r')\b'
        patrones[key] = re.compile(pattern_str, re.IGNORECASE)
    return patrones



def regex_region(df, patrones_listos):
    """
    Para poder inferir la región de la noticia se implementa 
    un sistema de puntuación donde la aparición de la región
    en el titulo vale 3 veces más que en el cuerpo (ya que 
    tiene mas incidencia en el contenido de la noticia).

    Luego la región resultante se elige por votación (es decir,
    la región con mas coincidencias en los aliases gana). En caso
    de empates, se decide por la que tuvo más puntaje en el titulo.
    En caso de un nuevo empate en este apartado, se escoge la región
    con menor id.
    """
    # Trabajamos sobre una copia temporal de columnas para no afectar el df original si algo falla
    score_cols = []

    for key, pattern in patrones_listos.items():
        col_name = f'score_{key}'
        
        score_titulo = df['title_clean'].str.count(pattern.pattern) * 3
        score_cuerpo = df['body_clean'].str.count(pattern.pattern)
        
        df[col_name] = (score_titulo + score_cuerpo) 
        score_cols.append(col_name)

    # Encontrar el puntaje máximo por fila
    max_scores = df[score_cols].max(axis=1)
    best_regions = df[score_cols].idxmax(axis=1).str.replace('score_', '').astype(int)
    df['region_id'] = np.where(max_scores < 1, 17, best_regions)
    df = df.drop(columns=score_cols)
    
    return df


"""
def regex_region(titulo, cuerpo, region_aliases):
    '''
    Para poder inferir la región de la noticia se implementa 
    un sistema de puntuación donde la aparición de la región
    en el titulo vale 3 veces más que en el cuerpo (ya que 
    tiene mas incidencia en el contenido de la noticia).

    Luego la región resultante se elige por votación (es decir,
    la región con mas coincidencias en los aliases gana). En caso
    de empates, se decide por la que tuvo más puntaje en el titulo.
    En caso de un nuevo empate en este apartado, se escoge la región
    con menor id.
    '''

    puntajes = {key: 0 for key in region_aliases.keys() if key != 17}

    for key, lista_alias in region_aliases.items():
        if key == 17: continue


        # asignación puntajes
        matches_titulo = len(pattern.findall(titulo))
        puntajes[key] += matches_titulo * 3
        matches_cuerpo = len(pattern.findall(cuerpo))
        puntajes[key] += matches_cuerpo * 1

    max_puntos = max(puntajes.values())

    if max_puntos > 0:
        return max(puntajes, key=puntajes.get)
    else:
        return 17 

"""
    

"\ndef regex_region(titulo, cuerpo, region_aliases):\n    '''\n    Para poder inferir la región de la noticia se implementa \n    un sistema de puntuación donde la aparición de la región\n    en el titulo vale 3 veces más que en el cuerpo (ya que \n    tiene mas incidencia en el contenido de la noticia).\n\n    Luego la región resultante se elige por votación (es decir,\n    la región con mas coincidencias en los aliases gana). En caso\n    de empates, se decide por la que tuvo más puntaje en el titulo.\n    En caso de un nuevo empate en este apartado, se escoge la región\n    con menor id.\n    '''\n\n    puntajes = {key: 0 for key in region_aliases.keys() if key != 17}\n\n    for key, lista_alias in region_aliases.items():\n        if key == 17: continue\n\n\n        # asignación puntajes\n        matches_titulo = len(pattern.findall(titulo))\n        puntajes[key] += matches_titulo * 3\n        matches_cuerpo = len(pattern.findall(cuerpo))\n        puntajes[key] += matches_cuerpo * 

In [6]:
def build_fact_table(df, dim_source, dim_region, dim_date):
    """
    Ensambla la Fact Table final uniendo los datos de las dimensiones
    creadas anteriormente; además de agregan recuentos en las columnas
    de body y title
    """

    # Recuentos de palabras en columnas pedidas
    #df['word_count_title'] = df['title_clean'].str.split().str.len()
    #df['word_count_body'] = df['body_clean'].str.split().str.len()

    df['word_count_title'] = df['title_clean'].str.count(r'\s+') + 1
    df['word_count_body'] = df['body_clean'].str.count(r'\s+') + 1

    
    # JOIN DF CON DIM SOURCE
    fact = df.merge(
        dim_source[['source_id', 'source_name']], 
        left_on='source', # key de df
        right_on='source_name', # key de la dim
        how='left'
    )
    
    # JOIN fact CON DIM DATE
    fact = fact.merge(
        dim_date[['date_id', 'full_date', 'year', 'month']], 
        left_on='publish_date', # key df
        right_on='full_date',  # key dim
        how='left'
    )

    # JOIN CON  dim region es implicito (ya esta la key cuando se hace la regex)
    fact = fact.rename(columns={'region_key': 'region_id'})
    fact_table = fact[['article_id', 'title', 'body', 'word_count_title', 'word_count_body',
        'date_id', 'source_id', 'region_id', 'year', 'month']].copy()
    
    return fact_table

![Diagrama Fact table y dimensiones](fact_dims_diagram.png)

## Validaciones

In [ ]:
def validar_consistencia_referencial(fact_news, dim_source, dim_date, dim_region):

    """
    Consistencia referencial: no deben existir llaves foraneas
    (o llaves subrogadas de dimensiones) huerfanas
    """
    source_ok = fact_news['source_id'].isin(dim_source['source_id']).all()
    region_ok = fact_news['region_id'].isin(dim_region['region_key']).all()
    date_ok = fact_news['date_id'].isin(dim_date['date_id']).all()
    
    return source_ok and region_ok and date_ok


def validar_conteo_filas(df_limpio, fact_news):
    """
    El conteo de filas en la tabla de hechos debe
    coincidir con el número de artículos crudos
    """
    return len(fact_news) == len(df_limpio)


def validar_unicidad_dimensiones(dim_source, dim_date, dim_region):
    """
    No deben existir llaves duplicadas dentro
    de una misma tabla de dimensiones
    """
    source_unique = dim_source['source_id'].is_unique
    date_unique = dim_date['date_id'].is_unique
    region_unique = dim_region['region_key'].is_unique
    
    return source_unique and date_unique and region_unique


def validar_particiones(fact_news):
    """ 
    La distribución de particiones debe ser correcta 
    (cada artículo en su partición correspondiente)
    """
    year_ok = fact_news['date_id'] // 10000 == fact_news['year']
    month_ok = (fact_news['date_id'] // 100 % 100) == fact_news['month'].astype(int)

    return (year_ok & month_ok).all()

def validar_word_count(fact_news):
    """
    consistencia en el word count, permite
    ver que no se hayan datos corruptos (que haga
    que el count cambie) despues de todo el proceso
    """

    title_wc_correcto = fact_news['word_count_title'] == fact_news['title'].str.count(r'\s+') + 1
    body_wc_correcto = fact_news['word_count_body'] == fact_news['body'].str.count(r'\s+') + 1
    
    return (title_wc_correcto & body_wc_correcto).all()

In [8]:
def run_validation(df_limpio, fact_news, dim_source, dim_date, dim_region):
    """
    Ejecutar todas las funciones de validaciones y comprobar
    si el proceso las pasa todas
    """

    return (
        validar_consistencia_referencial(fact_news, dim_source, dim_date, dim_region)
        and validar_conteo_filas(df_limpio, fact_news)
        and validar_unicidad_dimensiones(dim_source, dim_date, dim_region)
        and validar_particiones(fact_news)
        and validar_word_count(fact_news)
    )

## EJECUCIÓN

In [ ]:
def procesar_etl(input_file):
    '''
    Ejecuta el pipeline construido, aplicando las funciones
    definidas anteriormente.
    '''

    print("S T A R T")
    # Cargar el dataset
    df = pd.read_csv(input_file)
    print("Df leído")
    
    # Limpiar y normalizar dataset
    df = limpiar_dataset(df)
    df['title_clean'] = df['title'].apply(normalizar_texto)
    df['body_clean'] = df['body'].apply(normalizar_texto)
    print("Df limpio y normalizado")

        
    # Construcción de dimensiones y fact table
    dim_date = build_date_dim(df)
    dim_source = build_source_dim(df)
    dim_region, region_aliases = dicc_regiones()
    print("Dimensiones creadas")
    
    patrones_listos = construir_patrones(region_aliases)
    df = regex_region(df, patrones_listos)

    print("REGEX LISTA")
    #df['region_key'] = df.apply(lambda x: regex_region(x['title_clean'], x['body_clean'], region_aliases), axis=1) # Aplicación de la regex para obtener la region de la noticia
    fact_news = build_fact_table(df, dim_source, dim_region, dim_date)
    print("tabla de hechos creada")

    # Guardar dimensiones y almacenamiento particionado
    os.makedirs("warehouse/dim_source", exist_ok=True)
    os.makedirs("warehouse/dim_region", exist_ok=True)
    os.makedirs("warehouse/dim_date", exist_ok=True)
    dim_date.to_parquet("warehouse/dim_date/dim_date.parquet", index=False)
    dim_source.to_parquet("warehouse/dim_source/dim_source.parquet", index=False)
    dim_region.to_parquet("warehouse/dim_region/dim_region.parquet", index=False) 
    print("almacenamiento de dimensiones completado")

    fact_news.to_parquet(
        "warehouse/fact_news", 
        partition_cols=["year", "month"], 
        engine="pyarrow", 
        index=False
    )
    print("almacenamiento particionado completado")


    # Validaciones
    valido = run_validation(df, fact_news, dim_source, dim_date, dim_region)

    if valido: print(f"ARCHIVO PROCESADO CORRECTAMENTE")

    else: print("Error en el procesamiento")

In [10]:
# Ejecución
if __name__ == "__main__":
    procesar_etl('noticias_chile_2023_2025_unique_article_id.csv')

INICIO!!!!!!!!!!!!!
Df leído
Df limpio y normalizado
Dimensiones creadas
REGEX LISTA
tabla de hechos creada
almacenamiento de dimensiones completado
almacenamiento particionado completado


KeyboardInterrupt: 